# Actividad 2 — Principio de Responsabilidad Única (SRP)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Monitoreo y Gestión de Cuartos Fríos (Cadena de Frío)

---



## 1. Ejemplo Incorrecto (Violando SRP)

En este primer intento creamos una clase `CuartoFrioTodero` que hace absolutamente todo:
- Lee los voltajes del sensor físico de temperatura.
- Revisa si la temperatura es peligrosa para las vacunas o alimentos.
- Hace las cuentas del costo de energía eléctrica por los kWh gastados.
- Guarda un historial de registros en una lista (simulando base de datos o archivo).
- Arma el mensaje y simula el envío del correo electrónico de alerta.



In [14]:
# Ejemplo de clase que hace de todo, este no cumple
import datetime

class CuartoFrioTodero:
    def __init__(self, id_cuarto: str, tarifa_kwh: float, potencia_compresor_kw: float, correo_contacto: str):
        self.id_cuarto: str = id_cuarto
        self.tarifa_kwh: float = tarifa_kwh
        self.potencia_compresor_kw: float = potencia_compresor_kw
        self.correo_contacto: str = correo_contacto
        self.temperatura_actual: float = 0.0
        self.logs_historial: list = []

    # 1. Tarea de hardware / sensor
    def leer_sensor_voltaje(self, voltaje: float, escala: float) -> float:
        # convierte voltaje del sensor a grados celsius
        self.temperatura_actual = round((voltaje * escala) - 50.0, 2)
        return self.temperatura_actual

    # 2. Tarea de reglas de negocio / rango de temperatura
    def verificar_estado_temperatura(self, temp_min: float, temp_max: float) -> str:
        if self.temperatura_actual < temp_min:
            return f"Alerta Frío: {self.temperatura_actual}°C está por debajo de {temp_min}°C"
        elif self.temperatura_actual > temp_max:
            return f"Alerta Calor: {self.temperatura_actual}°C supera el límite de {temp_max}°C"
        return "Temperatura dentro de rango normal"

    # 3. Tarea contable / costo de energía
    def calcular_costo_luz(self, horas_uso: float) -> float:
        consumo_kwh = self.potencia_compresor_kw * horas_uso
        total_pesos = consumo_kwh * self.tarifa_kwh
        return round(total_pesos, 2)

    # 4. Tarea de persistencia / logs
    def guardar_log(self, mensaje_estado: str) -> str:
        fecha_hora = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        registro = f"[{fecha_hora}] Cuarto: {self.id_cuarto} | Temp: {self.temperatura_actual}°C | {mensaje_estado}"
        self.logs_historial.append(registro)
        return registro

    # 5. Tarea de comunicación / envío de correo
    def enviar_correo_emergencia(self, mensaje_alerta: str) -> str:
        asunto = f"URGENTE: Problema en {self.id_cuarto}"
        cuerpo = f"Para: {self.correo_contacto}\nAsunto: {asunto}\nDetalle: {mensaje_alerta}\nFavor revisar el compresor."
        return f"[Simulación SMTP enviada]\n{cuerpo}"




In [10]:
# Probamos la clase todera
print("--- Probando CuartoFrioTodero  ---")
cf_monolito = CuartoFrioTodero(
    id_cuarto="El Cuarto de vacunas de covid",
    tarifa_kwh=650.0,
    potencia_compresor_kw=12.0,
    correo_contacto="mantenimiento@bodegafrio.com"
)

# 1. Leemos sensor
temp = cf_monolito.leer_sensor_voltaje(voltaje=5.85, escala=10.0)
print(f"Temperatura: {temp}°C")

# 2. Verificamos si está en rango para vacunas: 2°C a 8°C
estado = cf_monolito.verificar_estado_temperatura(temp_min=2.0, temp_max=8.0)
print(f"Estado del producto: {estado}")

# 3. Calculamos la luz del día 24 horas
gasto_luz = cf_monolito.calcular_costo_luz(horas_uso=24.0)
print(f"Gasto estimado de luz (24h): ${gasto_luz:,.2f} COP")

# 4. Guardamos log
log = cf_monolito.guardar_log(estado)
print(f"Log guardado: {log}")

# 5. Mandamos correo
envio = cf_monolito.enviar_correo_emergencia(estado)
print(f"Correo:\n{envio}")


--- Probando CuartoFrioTodero  ---
Temperatura: 8.5°C
Estado del producto: Alerta Calor: 8.5°C supera el límite de 8.0°C
Gasto estimado de luz (24h): $187,200.00 COP
Log guardado: [2026-08-28 20:33:55] Cuarto: El Cuarto de vacunas de covid | Temp: 8.5°C | Alerta Calor: 8.5°C supera el límite de 8.0°C
Correo:
[Simulación SMTP enviada]
Para: mantenimiento@bodegafrio.com
Asunto: URGENTE: Problema en El Cuarto de vacunas de covid
Detalle: Alerta Calor: 8.5°C supera el límite de 8.0°C
Favor revisar el compresor.


## 2. Ejemplo Correcto (Aplicando SRP)

Para solucionar esto, dividimos el problema en **5 clases separadas**, donde cada una tiene una sola responsabilidad clara:

1. `SensorTemperatura`: Solo se encarga de tomar la lectura del sensor y calibrar el offset.
2. `ControladorRangoTermico`: Solo valida si la temperatura está dentro del rango seguro para el producto.
3. `CalculadorGastoEnergetico`: Solo calcula los kWh y el costo de energía según el compresor y las horas de uso.
4. `HistorialTelemetria`: Solo guarda los registros y permite consultar los últimos eventos.
5. `NotificadorEmergencia`: Solo se encarga de formatear y despachar las alertas por correo.


In [13]:
# Clases modulares aplicando SRP
import datetime

# lectura y calibración del sensor
class SensorTemperatura:
    def __init__(self, id_sensor: str, offset_calibracion: float = 0.0) -> None:
        self.id_sensor: str = id_sensor
        self.offset_calibracion: float = offset_calibracion

    def leer_celsius(self, voltaje: float, factor: float = 10.0) -> float:
        # calculo base de voltaje a temperatura
        temp_cruda = (voltaje * factor) - 50.0
        return round(temp_cruda + self.offset_calibracion, 2)

    def calibrar(self, nuevo_offset: float) -> None:
        self.offset_calibracion = nuevo_offset


#  validar si la temperatura es segura
class ControladorRangoTermico:
    def __init__(self, temp_min: float, temp_max: float) -> None:
        self.temp_min: float = temp_min
        self.temp_max: float = temp_max

    def esta_en_rango(self, temp: float) -> bool:
        return self.temp_min <= temp <= self.temp_max

    def evaluar_situacion(self, temp: float) -> str:
        if temp < self.temp_min:
            return f"Alerta Frío: {temp}°C está por debajo del mínimo ({self.temp_min}°C)"
        elif temp > self.temp_max:
            return f"Alerta Calor: {temp}°C supera el límite ({self.temp_max}°C)"
        return "Temperatura Normal y Estable"


# cálculo de consumo y costos de energía
class CalculadorGastoEnergetico:
    def __init__(self, tarifa_kwh: float, potencia_kw: float) -> None:
        self.tarifa_kwh: float = tarifa_kwh
        self.potencia_kw: float = potencia_kw

    def calcular_kwh(self, horas: float, factor_uso: float = 0.85) -> float:
        # el compresor no está al 100% todo el tiempo, modula según ciclo
        return round(self.potencia_kw * factor_uso * horas, 2)

    def calcular_costo_periodo(self, horas: float, factor_uso: float = 0.85) -> float:
        kwh = self.calcular_kwh(horas, factor_uso)
        return round(kwh * self.tarifa_kwh, 2)


#  guardar historial de lecturas
class HistorialTelemetria:
    def __init__(self, id_cuarto: str) -> None:
        self.id_cuarto: str = id_cuarto
        self.registros: list = []

    def agregar_lectura(self, temp: float, estado: str) -> str:
        hora_actual = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        linea = f"[{hora_actual}] [{self.id_cuarto}] Temp: {temp}°C -> {estado}"
        self.registros.append(linea)
        return linea

    def total_registros(self) -> int:
        return len(self.registros)


# armar y mandar alertas
class NotificadorEmergencia:
    def __init__(self, correo_tecnico: str, remitente: str = "sistema@bodegafrio.com") -> None:
        self.correo_tecnico: str = correo_tecnico
        self.remitente: str = remitente

    def armar_mensaje(self, id_cuarto: str, temp: float, detalle: str) -> str:
        return (
            f"De: {self.remitente}\n"
            f"Para: {self.correo_tecnico}\n"
            f"Asunto: ALERTA TÉRMICA EN {id_cuarto}\n"
            f"Mensaje: Se detectó {temp}°C. Detalle: {detalle}"
        )

    def enviar_aviso(self, id_cuarto: str, temp: float, detalle: str) -> str:
        texto = self.armar_mensaje(id_cuarto, temp, detalle)
        return f"[Correo enviado con éxito]\n{texto}"




In [ ]:
# Prueba
print("---  Sistema Modular ---\n")

# Instanciamos cada objeto por separado
sensor = SensorTemperatura(id_sensor="PT100-01", offset_calibracion=-0.2)
control_temp = ControladorRangoTermico(temp_min=2.0, temp_max=8.0)
calculador_luz = CalculadorGastoEnergetico(tarifa_kwh=650.0, potencia_kw=12.0)
historial = HistorialTelemetria(id_cuarto="El cuarto de vacunas de covid")
notificador = NotificadorEmergencia(correo_tecnico="mantenimiento@bodegafrio.com")

# 1 Tomamos lectura con el sensor
temperatura = sensor.leer_celsius(voltaje=5.85, factor=10.0)
print(f"1. Sensor ({sensor.id_sensor}): {temperatura}°C")

# 2 Revisamos si es segura
es_segura = control_temp.esta_en_rango(temperatura)
diagnostico = control_temp.evaluar_situacion(temperatura)
print(f"2. Diagnóstico: {diagnostico} (En rango?: {es_segura})")

# 3 Calculamos la luz del día
kwh_dia = calculador_luz.calcular_kwh(horas=24)
costo_dia = calculador_luz.calcular_costo_periodo(horas=24)
print(f"3. Consumo 24h: {kwh_dia} kWh | Costo estimado: ${costo_dia:,.2f} COP")

# 4 Guardamos en el historial
log_guardado = historial.agregar_lectura(temperatura, diagnostico)
print(f"4. Registro: {log_guardado} (Total logs: {historial.total_registros()})")

# 5 Si no está en rango, disparamos la alerta
if not es_segura:
    aviso = notificador.enviar_aviso("El cuarto de vacunas de covid", temperatura, diagnostico)
    print(f"5. Notificación:\n{aviso}")


---  Sistema Modular ---

1. Sensor (PT100-01): 8.3°C
2. Diagnóstico: Alerta Calor: 8.3°C supera el límite (8.0°C) (¿En rango?: False)
3. Consumo 24h: 244.8 kWh | Costo estimado: $159,120.00 COP
4. Registro: [2026-08-28 20:37:43] [El cuarto de vacunas de covid] Temp: 8.3°C -> Alerta Calor: 8.3°C supera el límite (8.0°C) (Total logs: 1)
5. Notificación:
[Correo enviado con éxito]
De: sistema@bodegafrio.com
Para: mantenimiento@bodegafrio.com
Asunto: ALERTA TÉRMICA EN El cuarto de vacunas de covid
Mensaje: Se detectó 8.3°C. Detalle: Alerta Calor: 8.3°C supera el límite (8.0°C)
